# Волга-IT 2026. Финал. «Искусственный интеллект и анализ данных»
## Интерактивный аналитический отчёт: Настроение в домовых чатах ЖКХ

Данный ноутбук воспроизводит ключевые шаги формирования аналитики и визуализаций:
1. **Загрузка и валидация результатов инференса** (`chat_summary.csv`, `chat_daily.csv`, `messages_pred.csv`).
2. **Сравнительный рейтинг напряжённости чатов** (динамика, тренды, пики конфликтов).
3. **Интерактивные визуализации (Plotly)**:
   - Временной ряд изменения индекса настроения жителей (`mood_index`).
   - Связь всплесков активности и доли агрессивных высказываний.
   - Тепловая карта настроений (Heatmap Чат × Дата) для мгновенного выявления проблемных дней.
   - Структура тональности обращений жителей (Stacked Bar).
   - Суточный профиль активности и конфликтности (почасовой срез ЖКХ).
4. **Глубокий анализ инцидентов (Deep Dive)**: автоматическое выделение тем и реплик жителей в дни-пики.
5. **Экспорт автономного HTML-отчёта** `report.html`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Импорт графических модулей и генератора отчёта
from src.report.charts import (
    build_mood_dynamics_figure,
    build_aggression_volume_figure,
    build_mood_heatmap_figure,
    build_sentiment_breakdown_figure,
    build_hourly_profile_figure,
)
from src.report.generator import generate_html_report, load_report_datasets, calculate_report_kpi

print("Все модули аналитики и визуализации успешно импортированы.")

### 1. Загрузка данных и результатов инференса
Загружаем сформированные файлы `chat_summary.csv`, `chat_daily.csv`, `messages_pred.csv` и объединяем с исходным текстом сообщений:

In [ ]:
pred_dir = Path("dev_out")
msgs_path = Path("data/dev_messages.csv")

summary_df, daily_df, messages_df = load_report_datasets(pred_dir, msgs_path)
kpi = calculate_report_kpi(summary_df, messages_df)

print(f"Всего домовых чатов:      {kpi['total_chats']}")
print(f"Всего сообщений:          {kpi['total_messages']}")
print(f"Реплик жителей:           {kpi['resident_msgs']}")
print(f"Служебных рассылок:       {kpi['service_msgs']} ({kpi['service_share']:.1f}%)")
print(f"Общий индекс настроения:  {kpi['global_mood']:.2f}")
print(f"Доля агрессии в чатах:    {kpi['global_agg_share']:.1f}%")

### 2. Сравнительный анализ домовых чатов
Таблица показателей по каждому чату:

In [ ]:
summary_view = summary_df.copy()
summary_view["mood_index"] = summary_view["mood_index"].round(2)
summary_view["negative_share"] = (summary_view["negative_share"] * 100).round(1).astype(str) + "%"
summary_view["aggression_share"] = (summary_view["aggression_share"] * 100).round(1).astype(str) + "%"
summary_view

### 3. Интерактивная динамика настроения жителей по дням (mood_index)
Линейный график отображает динамику индекса социального самочувствия. Линия нуля разделяет позитив и негатив:

In [ ]:
fig_mood = build_mood_dynamics_figure(daily_df)
fig_mood.show()

### 4. Всплески активности и уровень агрессии
Сопоставление общего количества сообщений жителей и средней доли агрессивных высказываний:

In [ ]:
fig_agg = build_aggression_volume_figure(daily_df)
fig_agg.show()

### 5. Тепловая карта настроений (Heatmap Чат × Дата)
Мгновенно визуализирует проблемные дни по всем домам (красные зоны соответствуют всплескам негатива):

In [ ]:
fig_heat = build_mood_heatmap_figure(daily_df)
fig_heat.show()

### 6. Структура обращений и суточный график активности

In [ ]:
fig_sent = build_sentiment_breakdown_figure(summary_df)
fig_sent.show()

fig_hour = build_hourly_profile_figure(messages_df)
fig_hour.show()

### 7. Экспорт автономного HTML-отчёта
Генерируем `report.html` для передачи жилищной инспекции и экспертному жюри:

In [ ]:
report_file = generate_html_report(
    pred_dir="dev_out",
    output_path="report.html",
    messages_path="data/dev_messages.csv",
)
print(f"Автономный HTML-отчёт успешно создан: {report_file.resolve()}")